In [1]:
# Imports
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    RepeatedKFold,
    RandomizedSearchCV
)
import numpy as np
import pandas as pd
import time

# Recherche du meilleur modèle de prévision
Environ 35 features, une target = débit, dataset total labelisé environ 60–100 lignes

In [2]:
# 1️⃣ Pipelines SANS réduction de dimensionnalité
# RobustScaler utilisé partout : résistant aux outliers (médiane + IQR)
# StandardScaler serait sensible aux valeurs extrêmes de RMS, flux, band_energy
def build_basic_pipelines():
    pipelines = {}

    pipelines["Ridge"] = Pipeline([
        ("scaler", RobustScaler()),
        ("model", Ridge(alpha=1.0))
    ])

    pipelines["Lasso"] = Pipeline([
        ("scaler", RobustScaler()),
        ("model", Lasso(alpha=0.01, max_iter=10000))
    ])

    # RandomForest est insensible au scaling (seuils, pas distances)
    # mais RobustScaler gardé pour cohérence du pipeline
    pipelines["RandomForest"] = Pipeline([
        ("scaler", RobustScaler()),
        ("model", RandomForestRegressor(random_state=42))
    ])

    # SVR : scaling critique — le noyau RBF calcule des distances
    pipelines["SVR"] = Pipeline([
        ("scaler", RobustScaler()),
        ("model", SVR(kernel="rbf", C=1.0, epsilon=0.1))
    ])

    # KNN : scaling critique — repose entièrement sur des distances
    pipelines["KNN"] = Pipeline([
        ("scaler", RobustScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5, weights="distance"))
    ])

    return pipelines

| Modèle       | Ce qu'il capte bien     | Sensible au scaling |
| ------------ | ----------------------- | ------------------- |
| Ridge        | relations globales      | Oui                 |
| Lasso        | sélection de features   | Oui                 |
| SVR          | relations non linéaires | Oui (critique)      |
| KNN          | similarité locale       | Oui (critique)      |
| RandomForest | interactions complexes  | Non                 |

In [3]:
# Modèle Stacking : combine Ridge + SVR + KNN avec un meta-modèle Ridge
# Défini séparément pour être réutilisable dans les pipelines PCA et SelectK
stack_model = StackingRegressor(
    estimators=[
        ("ridge", Ridge()),
        ("svr",   SVR(kernel="rbf", C=1.0, epsilon=0.1)),
        ("knn",   KNeighborsRegressor(n_neighbors=3))
    ],
    final_estimator=Ridge()
)

In [4]:
# 2️⃣ Pipelines AVEC PCA
# n_components=0.97 → conserve les composantes qui expliquent 97% de la variance
# Réduit la redondance entre features acoustiques corrélées (centroid, rolloff, bandwidth...)
#
# Ce que fait le pipeline à l'entraînement :
#     RobustScaler.fit_transform(X_train)
#     PCA.fit_transform(X_train)
#     Model.fit(...)
#
# En prédiction / scoring :
#     RobustScaler.transform(X_test)
#     PCA.transform(X_test)
#     Model.predict(...)

def build_pca_pipelines(n_components=0.97):
    pipelines = {}

    pipelines["Ridge_PCA"] = Pipeline([
        ("scaler", RobustScaler()),
        ("pca",    PCA(n_components=n_components)),
        ("model",  Ridge(alpha=1.0))
    ])

    pipelines["SVR_PCA"] = Pipeline([
        ("scaler", RobustScaler()),
        ("pca",    PCA(n_components=n_components)),
        ("model",  SVR(kernel="rbf", C=1.0, epsilon=0.1))
    ])

    pipelines["KNN_PCA"] = Pipeline([
        ("scaler", RobustScaler()),
        ("pca",    PCA(n_components=n_components)),
        ("model",  KNeighborsRegressor(n_neighbors=5, weights="distance"))
    ])

    # RandomForest avec PCA : perd l'interprétabilité des features
    # mais peut réduire le bruit si les features sont très redondantes
    # → à comparer avec RandomForest sans PCA dans les résultats
    pipelines["RandomForest_PCA"] = Pipeline([
        ("scaler", RobustScaler()),
        ("pca",    PCA(n_components=n_components)),
        ("model",  RandomForestRegressor(random_state=42))
    ])

    pipelines["Stacked_PCA"] = Pipeline([
        ("scaler", RobustScaler()),
        ("pca",    PCA(n_components=n_components)),
        ("model",  stack_model)
    ])

    return pipelines

In [5]:
# 3️⃣ Pipelines avec SelectKBest
# Sélectionne les k features les plus corrélées à la target (débit)
# via le score F de régression linéaire
# Avantage sur PCA : conserve les features originales → interprétables
# Inconvénient : ne capte pas les corrélations entre features
def build_selectkbest_pipelines(k=15):
    pipelines = {}

    pipelines["Ridge_SelectK"] = Pipeline([
        ("scaler", RobustScaler()),
        ("select", SelectKBest(score_func=f_regression, k=k)),
        ("model",  Ridge(alpha=1.0))
    ])

    pipelines["SVR_SelectK"] = Pipeline([
        ("scaler", RobustScaler()),
        ("select", SelectKBest(score_func=f_regression, k=k)),
        ("model",  SVR(kernel="rbf", C=1.0, epsilon=0.1))
    ])

    pipelines["KNN_SelectK"] = Pipeline([
        ("scaler", RobustScaler()),
        ("select", SelectKBest(score_func=f_regression, k=k)),
        ("model",  KNeighborsRegressor(n_neighbors=5, weights="distance"))
    ])

    return pipelines

In [6]:
# 4️⃣ Construction de tous les pipelines
def build_all_pipelines():
    pipelines = {}
    pipelines.update(build_basic_pipelines())
    pipelines.update(build_pca_pipelines(n_components=0.97))
    pipelines.update(build_selectkbest_pipelines(k=15))
    return pipelines

In [7]:
# ==========================================
# 🔹 CONFIGURATION HYPERPARAMÈTRES
# ==========================================

def get_param_grid():
    """
    Centralise toutes les grilles d'hyperparamètres.
    Les clés correspondent exactement aux noms des pipelines
    retournés par build_all_pipelines().
    """
    return {
        "KNN": {
            "model__n_neighbors": [3, 5, 7],
            "model__weights"    : ["uniform", "distance"],
            "model__p"          : [1, 2]
        },
        "KNN_PCA": {
            "model__n_neighbors": [3, 5, 7],
            "model__weights"    : ["uniform", "distance"],
            "model__p"          : [1, 2]
        },
        "KNN_SelectK": {
            "model__n_neighbors": [3, 5, 7],
            "model__weights"    : ["uniform", "distance"],
            "model__p"          : [1, 2]
        },
        "Ridge": {
            "model__alpha" : [0.01, 0.1, 1.0, 10.0],
            "model__solver": ["auto", "svd"]
        },
        "Ridge_PCA": {
            "model__alpha" : [0.01, 0.1, 1.0, 10.0],
            "model__solver": ["auto", "svd"]
        },
        "Ridge_SelectK": {
            "model__alpha" : [0.01, 0.1, 1.0, 10.0],
            "model__solver": ["auto", "svd"]
        },
        "Lasso": {
            "model__alpha"    : [0.001, 0.01, 0.1, 1.0],
            "model__max_iter" : [5000, 10000],
            "model__selection": ["cyclic", "random"]
        },
        "RandomForest": {
            "model__n_estimators"     : [50, 100, 200],
            "model__max_depth"        : [None, 3, 5, 10],
            "model__min_samples_split": [2, 4, 6],
            "model__min_samples_leaf" : [1, 2, 3],
            "model__max_features"     : ["sqrt", "log2"]
        },
        "RandomForest_PCA": {
            "model__n_estimators"     : [50, 100, 200],
            "model__max_depth"        : [None, 3, 5, 10],
            "model__min_samples_split": [2, 4, 6],
            "model__min_samples_leaf" : [1, 2, 3],
            "model__max_features"     : ["sqrt", "log2"]
        },
        "SVR": {
            "model__C"      : [0.1, 1.0, 10.0, 100.0],
            "model__epsilon": [0.01, 0.1, 0.5],
            "model__kernel" : ["rbf", "linear"],
            "model__gamma"  : ["scale", "auto"]
        },
        "SVR_PCA": {
            "model__C"      : [0.1, 1.0, 10.0, 100.0],
            "model__epsilon": [0.01, 0.1, 0.5],
            "model__kernel" : ["rbf", "linear"],
            "model__gamma"  : ["scale", "auto"]
        },
        "SVR_SelectK": {
            "model__C"      : [0.1, 1.0, 10.0, 100.0],
            "model__epsilon": [0.01, 0.1, 0.5],
            "model__kernel" : ["rbf", "linear"],
            "model__gamma"  : ["scale", "auto"]
        },
    }


def get_n_iter_map():
    """
    n_iter adapté à la taille de l'espace de recherche de chaque modèle.
    Règle : si nb_combinaisons <= 20 → exhaustif, sinon ~25% de l'espace.
    """
    return {
        "KNN"             : 12,   # 3×2×2 = 12   → exhaustif
        "KNN_PCA"         : 12,
        "KNN_SelectK"     : 12,
        "Ridge"           : 8,    # 4×2   =  8   → exhaustif
        "Ridge_PCA"       : 8,
        "Ridge_SelectK"   : 8,
        "Lasso"           : 16,   # 4×2×2 = 16   → exhaustif
        "RandomForest"    : 50,   # 216 combinaisons → ~25%
        "RandomForest_PCA": 50,
        "SVR"             : 24,   # 48 combinaisons  → 50%
        "SVR_PCA"         : 24,
        "SVR_SelectK"     : 24,
    }




# Benchmark avec cross-validation

Pour chaque modèle, calcul :

✅ MAE moyen en CV (robuste)  
✅ Variabilité (std)  
✅ MAE sur test réel  
✅ R² sur test réel  
✅ Temps d'exécution  
✅ Nombre de composantes PCA (si applicable)

# 🔒 Zéro data leakage

Avec 50–100 samples :
- Le R² test peut beaucoup varier selon le split
- Le MAE CV est plus stable car moyenné sur 5 folds

👉 Priorité au **CV_MAE_mean** pour choisir le modèle  
👉 **Test_R2** et **Test_MAE** pour validation finale seulement

In [8]:
def benchmark_models(X, y):
    """
    Benchmark avec split train/test (80/20) + cross-validation sur train.
    Retourne un DataFrame trié par CV_MAE_mean croissant.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    pipelines = build_all_pipelines()
    results   = []

    for name, pipe in pipelines.items():

        # Cross-validation sur le train uniquement (pas de leakage)
        cv_scores = cross_val_score(
            pipe, X_train, y_train,
            cv=5,
            scoring="neg_mean_absolute_error"
        )
        mae_mean = -np.mean(cv_scores)
        mae_std  =  np.std(cv_scores)

        # Entraînement final sur tout le train pour évaluation sur test
        pipe.fit(X_train, y_train)

        # Score test
        start_time = time.time()
        y_pred     = pipe.predict(X_test)
        test_time  = time.time() - start_time

        r2_test  = pipe.score(X_test, y_test)
        mae_test = mean_absolute_error(y_test, y_pred)

        # Nombre de composantes PCA retenues (si applicable)
        n_pca_components = None
        if "pca" in pipe.named_steps:
            n_pca_components = pipe.named_steps["pca"].n_components_

        results.append({
            "Model"          : name,
            "CV_MAE_mean"    : round(mae_mean, 4),
            "CV_MAE_std"     : round(mae_std,  4),
            "Test_MAE"       : round(mae_test, 4),
            "Test_R2"        : round(r2_test,  4),
            "Test_time_sec"  : round(test_time, 4),
            "PCA_components" : n_pca_components
        })

    return (
        pd.DataFrame(results)
        .sort_values("CV_MAE_mean")
        .reset_index(drop=True)
    )

# Benchmark Benchmark avec split train/test + variation hyperparamètres
via RandomizedSearchCV.

In [9]:
def benchmark_models_tuned(X, y, test_size=0.2, random_state=42):
    """
    Benchmark train/test + CV avec tuning des hyperparamètres
    via RandomizedSearchCV.

    Même split que benchmark_models() si random_state identique
    → permet comparaison directe tuned vs non-tuned.

    Retourne un DataFrame trié par CV_MAE_mean croissant.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    pipelines   = build_all_pipelines()
    param_grid  = get_param_grid()
    n_iter_map  = get_n_iter_map()
    results     = []

    for name, pipeline in pipelines.items():

        if name not in param_grid:
            print(f"  ⏭️  {name} — pas de param_grid, skip.")
            continue

        print(f"  🔧 Tuning {name}...")

        rs = RandomizedSearchCV(
            estimator           = pipeline,
            param_distributions = param_grid[name],
            n_iter              = n_iter_map.get(name, 20),
            scoring             = "neg_mean_absolute_error",
            cv                  = 5,
            n_jobs              = -1,
            random_state        = random_state
        )
        rs.fit(X_train, y_train)

        cv_mae_mean = -rs.best_score_
        cv_mae_std  =  rs.cv_results_["std_test_score"][rs.best_index_]

        y_pred   = rs.best_estimator_.predict(X_test)
        mae_test = mean_absolute_error(y_test, y_pred)
        r2_test  = rs.best_estimator_.score(X_test, y_test)

        n_pca = None
        if "pca" in rs.best_estimator_.named_steps:
            n_pca = rs.best_estimator_.named_steps["pca"].n_components_

        results.append({
            "Model"         : name,
            "CV_MAE_mean"   : round(cv_mae_mean, 4),
            "CV_MAE_std"    : round(cv_mae_std,  4),
            "Test_MAE"      : round(mae_test,    4),
            "Test_R2"       : round(r2_test,     4),
            "PCA_components": n_pca,
            "Best_Params"   : rs.best_params_,
            "Best_Estimator": rs.best_estimator_
        })

    return (
        pd.DataFrame(results)
        .sort_values("CV_MAE_mean")
        .reset_index(drop=True)
    )

# Benchmark full CV (RepeatedKFold) — recommandé pour petits datasets

Quand le dataset est petit (< 100 lignes), le split train/test est risqué :
un seul split peut tomber sur une partition favorable ou défavorable par hasard.

RepeatedKFold avec 5 folds × 10 répétitions = **50 évaluations** différentes
→ estimation bien plus stable du MAE réel du modèle.

In [10]:
def benchmark_models_RepeatedKFold(X, y):
    """
    Benchmark full cross-validation sans split train/test.
    Utilise RepeatedKFold (5 folds × 10 répétitions = 50 scores).
    Recommandé quand n_samples < 100.
    Retourne un DataFrame trié par MAE_mean croissant.
    """
    pipelines = build_all_pipelines()
    results   = []

    rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

    for name, model in pipelines.items():

        scores     = cross_val_score(
            model, X, y,
            cv=rkf,
            scoring="neg_mean_absolute_error"
        )
        mae_scores = -scores

        results.append({
            "Model"   : name,
            "MAE_mean": round(np.mean(mae_scores), 4),
            "MAE_std" : round(np.std(mae_scores),  4)
        })

    return (
        pd.DataFrame(results)
        .sort_values("MAE_mean")
        .reset_index(drop=True)
    )

# Code principal
Lancement du benchmark sur la base du fichier CSV des données de calibration

In [11]:
# ==========================================
# 🔹 CODE PRINCIPAL
# ==========================================

# 1️⃣ Charger le dataset
calib = pd.read_csv("final_calibration_sounds_features.csv", sep=",")

# 2️⃣ Séparer features / target
X = calib.iloc[:, :-1].values
y = calib.iloc[:, -1].values

print("Shape dataset :", calib.shape)
print("Nb features   :", X.shape[1])
print("Nb samples    :", X.shape[0])
print("Target        :", calib.columns[-1])
print("-" * 50)


# ==========================================
# 🔹 BENCHMARK CLASSIQUE (train/test + CV)
# ==========================================

print("🔎 Benchmark avec split train/test + CV")
results_df = benchmark_models(X, y)
print("\nRésultats triés par CV_MAE_mean :")
print(results_df.to_string(index=False))


# ==========================================
# 🔹 BENCHMARK FULL CV (petits datasets)
# ==========================================

print("\n🔎 Benchmark full Cross-Validation (RepeatedKFold 5×10)")
results_cv_df = benchmark_models_RepeatedKFold(X, y)
print("\nRésultats triés par MAE_mean :")
print(results_cv_df.to_string(index=False))

# ── Benchmark avec tuning ──────────────────────────────────────
print("\n🔎 Benchmark avec tuning des hyperparamètres")
results_tuned_df = benchmark_models_tuned(X, y)
print(results_tuned_df[["Model", "CV_MAE_mean", "CV_MAE_std",
                          "Test_MAE", "Test_R2",
                          "PCA_components"]].to_string(index=False))

# ── Comparaison synthétique tuned vs défauts ──────────────────
print("\n📊 Gain du tuning (CV_MAE_mean) :")
comparison = results_df[["Model", "CV_MAE_mean"]].merge(
    results_tuned_df[["Model", "CV_MAE_mean"]],
    on="Model", suffixes=("_default", "_tuned"), how="inner"
)
comparison["Gain"]  = (comparison["CV_MAE_mean_default"] - comparison["CV_MAE_mean_tuned"]).round(4)
comparison["Gain%"] = (comparison["Gain"] / comparison["CV_MAE_mean_default"] * 100).round(1)
print(comparison.sort_values("Gain", ascending=False).to_string(index=False))

Shape dataset : (792, 43)
Nb features   : 42
Nb samples    : 792
Target        : debit
--------------------------------------------------
🔎 Benchmark avec split train/test + CV

Résultats triés par CV_MAE_mean :
           Model  CV_MAE_mean  CV_MAE_std  Test_MAE  Test_R2  Test_time_sec  PCA_components
    RandomForest       0.9721      0.1099    0.9221   0.9370         0.0093             NaN
     KNN_SelectK       1.0551      0.1024    0.8802   0.9511         0.0040             NaN
             KNN       1.1966      0.1170    1.1206   0.9295         0.0189             NaN
         KNN_PCA       1.2252      0.1386    1.1797   0.9183         0.0526            23.0
     Stacked_PCA       1.2504      0.1231    1.1778   0.9262         0.0056            23.0
RandomForest_PCA       1.5195      0.1989    1.4505   0.8946         0.0082            23.0
     SVR_SelectK       1.5980      0.1443    1.4590   0.8821         0.0086             NaN
             SVR       1.6545      0.1011    1.5835 

In [12]:
# ── Récupérer le meilleur modèle global (1er du classement) ───
best_row       = results_tuned_df.iloc[0]
best_model_name      = best_row["Model"]
best_params          = best_row["Best_Params"]
best_estimator       = best_row["Best_Estimator"]  # pipeline complet prêt à l'emploi

# print(f"🏆 Meilleur modèle : {best_model_name}")
# print(f"   CV_MAE_mean    : {best_row['CV_MAE_mean']}")
# print(f"   Test_MAE       : {best_row['Test_MAE']}")
# print(f"   Hyperparamètres: {best_params}")

# ── Récupérer les 3 meilleurs modèles du classement ───────────
TOP_N = 3

for i, row in results_tuned_df.head(TOP_N).iterrows():
    print(f"\n🏆 #{i+1} — {row['Model']}")
    print(f"   CV_MAE_mean    : {row['CV_MAE_mean']}")
    print(f"   Test_MAE       : {row['Test_MAE']}")
    print(f"   Hyperparamètres: {row['Best_Params']}")
    
# ── Récupérer les hyperparamètres d'un modèle précis ──────────
def get_best_params(results_df, model_name):
    """Retourne les meilleurs hyperparamètres d'un modèle par son nom."""
    row = results_df[results_df["Model"] == model_name]
    if row.empty:
        print(f"⚠️  Modèle '{model_name}' non trouvé")
        return None, None
    return row.iloc[0]["Best_Params"], row.iloc[0]["Best_Estimator"]


🏆 #1 — KNN_SelectK
   CV_MAE_mean    : 0.8954
   Test_MAE       : 0.8133
   Hyperparamètres: {'model__weights': 'distance', 'model__p': 1, 'model__n_neighbors': 3}

🏆 #2 — KNN
   CV_MAE_mean    : 0.9371
   Test_MAE       : 0.9073
   Hyperparamètres: {'model__weights': 'distance', 'model__p': 1, 'model__n_neighbors': 3}

🏆 #3 — SVR
   CV_MAE_mean    : 1.0316
   Test_MAE       : 1.0075
   Hyperparamètres: {'model__kernel': 'rbf', 'model__gamma': 'scale', 'model__epsilon': 0.1, 'model__C': 100.0}


In [13]:
# Exemple d'utilisation
params_rf, estimator_rf = get_best_params(results_tuned_df, "RandomForest")
params_svr, estimator_svr = get_best_params(results_tuned_df, "SVR")

print(f"\n RandomForest — meilleurs params : {params_rf}")
print(f"\n SVR          — meilleurs params : {params_svr}")


 RandomForest — meilleurs params : {'model__n_estimators': 100, 'model__min_samples_split': 6, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': None}

 SVR          — meilleurs params : {'model__kernel': 'rbf', 'model__gamma': 'scale', 'model__epsilon': 0.1, 'model__C': 100.0}


Phase 1 — Évaluation  : train (80%) + test (20%)  → choisir modèle et params

Phase 2 — Production  : tout le dataset (100%)     → modèle final à déployer

Le point clé sur set_params(**best_params)
best_estimator a été fitté sur 80% des données — on ne peut pas juste le re-fitter, sklearn ne permet pas de changer les données d'un modèle déjà fitté proprement. La bonne approche est de reconstruire le pipeline via build_all_pipelines()[best_name], d'y injecter les meilleurs hyperparamètres avec set_params(), puis de fitter sur 100% des données.

In [ ]:
# ==========================================
# 🔹 ENTRAÎNEMENT FINAL SUR TOUT LE DATASET
# ==========================================
# Les hyperparamètres ont été choisis via benchmark_models_tuned()
# → on peut maintenant utiliser 100% des données

# ── Option 1 : prendre automatiquement le meilleur du classement ──
best_row    = results_tuned_df.iloc[0]
best_name   = best_row["Model"]
best_params = best_row["Best_Params"]

# ── Option 2 : choisir manuellement le modèle ─────────────────────
# best_name   = "RandomForest"
# best_params, _ = get_best_params(results_tuned_df, best_name)

# ── Option 3 : choisir manuellement le modèle ET les paramètres ───
# best_name   = "RandomForest"
# best_params = {
#     "model__n_estimators"     : 200,
#     "model__max_depth"        : None,
#     "model__min_samples_split": 2,
#     "model__min_samples_leaf" : 1,
#     "model__max_features"     : "sqrt"
# }

print(f"🏆 Modèle choisi   : {best_name}")
print(f"   Hyperparamètres : {best_params}")
print(f"   CV_MAE_mean     : {best_row['CV_MAE_mean']} mL/s")
print(f"   Test_MAE        : {best_row['Test_MAE']} mL/s")

# Reconstruire le pipeline avec les meilleurs hyperparamètres
# On ne réutilise PAS best_estimator directement car il a été
# fitté sur X_train (80%) — on recrée et refit sur 100%
best_pipeline_full = build_all_pipelines()[best_name]
best_pipeline_full.set_params(**best_params)

# Entraînement sur 100% du dataset
best_pipeline_full.fit(X, y)

print(f"\n✅ Modèle final entraîné sur {len(X)} samples (100%)")

# Vérification sanité : MAE sur tout le dataset
# (sera optimiste par construction — juste pour vérifier qu'il n'y a pas de bug)
y_pred_train = best_pipeline_full.predict(X)
mae_train    = mean_absolute_error(y, y_pred_train)
print(f"   MAE sur dataset complet (optimiste) : {mae_train:.4f} mL/s")
print(f"   MAE de référence (CV honest)        : {best_row['CV_MAE_mean']} mL/s")

# Sauvegarde du modèle final
import joblib
pkl_path = f"final_model_{best_name}.pkl"
joblib.dump(best_pipeline_full, pkl_path)
print(f"\n💾 Modèle final sauvegardé : {pkl_path}")

🏆 Modèle choisi   : KNN_SelectK
   Hyperparamètres : {'model__weights': 'distance', 'model__p': 1, 'model__n_neighbors': 3}
   CV_MAE_mean     : 0.8954 mL/s
   Test_MAE        : 0.8133 mL/s

✅ Modèle final entraîné sur 792 samples (100%)
   MAE sur dataset complet (optimiste) : 0.0000 mL/s
   MAE de référence (CV honest)        : 0.8954 mL/s

💾 Modèle final sauvegardé : final_model_KNN_SelectK.pkl


tests a supprimer?

In [14]:
# diagnostic ENTRE les deux benchmarks

print("=== DIAGNOSTIC X et y ===")
print(f"type(X)      : {type(X)}")
print(f"X.shape      : {X.shape}")
print(f"y.shape      : {y.shape}")
print(f"y.mean()     : {y.mean():.4f}")
print(f"y.min()      : {y.min():.4f}")
print(f"y.max()      : {y.max():.4f}")
print(f"y.std()      : {y.std():.4f}")
print()

# Test manuel RandomForest avec paramètres par défaut
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
import numpy as np

pipe_test = Pipeline([
    ("scaler", RobustScaler()),
    ("model",  RandomForestRegressor(random_state=42))
])

scores = cross_val_score(
    pipe_test, X, y,
    cv=5,
    scoring="neg_mean_absolute_error"
)
print(f"RF par défaut sur X/y actuels — MAE CV : {-scores.mean():.4f} ± {scores.std():.4f}")

=== DIAGNOSTIC X et y ===
type(X)      : <class 'numpy.ndarray'>
X.shape      : (792, 42)
y.shape      : (792,)
y.mean()     : 11.2276
y.min()      : 0.0000
y.max()      : 25.4450
y.std()      : 5.7276

RF par défaut sur X/y actuels — MAE CV : 4.1213 ± 2.8166


In [15]:
# ==========================================
# 🔹 OPTIMISATION DES HYPERPARAMÈTRES (RandomizedSearchCV)
# ==========================================
# RandomizedSearchCV est préférable à GridSearchCV sur petit dataset :
# tire n_iter combinaisons aléatoires au lieu de tester exhaustivement
# → beaucoup plus rapide, résultats comparables
#
# Note : tester 3 jeux de features extraites à sr différents (16, 32, 44.1 kHz)
# pour valider la robustesse inter-téléphones

df = pd.read_csv("final_calibration_sounds_features.csv", sep=",")
X  = df.iloc[:, :-1].values
y  = df.iloc[:, -1].values

pipelines = build_all_pipelines()

# Grilles d'hyperparamètres adaptées aux petits datasets
param_grid = {
    "KNN": {
        "model__n_neighbors": [3, 5, 7],
        "model__weights"    : ["uniform", "distance"],
        "model__p"          : [1, 2]
    },
    "Ridge": {
        "model__alpha" : [0.01, 0.1, 1.0, 10.0],
        "model__solver": ["auto", "svd"]
    },
    "Lasso": {
        "model__alpha"    : [0.001, 0.01, 0.1, 1.0],
        "model__max_iter" : [5000, 10000],
        "model__selection": ["cyclic", "random"]
    },
    "RandomForest": {
        "model__n_estimators"     : [50, 100, 200],
        "model__max_depth"        : [None, 3, 5, 10],
        "model__min_samples_split": [2, 4, 6],
        "model__min_samples_leaf" : [1, 2, 3],
        "model__max_features"     : ["sqrt", "log2"]   # "auto" déprécié depuis sklearn 1.1
    },
    "RandomForest_PCA": {
        "model__n_estimators"     : [50, 100, 200],
        "model__max_depth"        : [None, 3, 5, 10],
        "model__min_samples_split": [2, 4, 6],
        "model__min_samples_leaf" : [1, 2, 3],
        "model__max_features"     : ["sqrt", "log2"]
    },
    "SVR": {
        "model__C"      : [0.1, 1.0, 10.0, 100.0],
        "model__epsilon": [0.01, 0.1, 0.5],
        "model__kernel" : ["rbf", "linear"],
        "model__gamma"  : ["scale", "auto"]
    }
}

results = []

# n_iter différent selon la complexité de l'espace de recherche
n_iter_map = {
    "KNN"            : 12,   # 3×2×2 = 12  → exhaustif
    "Ridge"          : 8,    # 4×2   =  8  → exhaustif
    "Lasso"          : 16,   # 4×2×2 = 16  → exhaustif
    "RandomForest"   : 50,   # 216 combinaisons → 50 tirages minimum
    "RandomForest_PCA": 50,
    "SVR"            : 24,   # 4×3×2×2 = 48 → 24 tirages
}

for name, pipeline in pipelines.items():
    print(f"\n===== RandomizedSearchCV pour {name} =====")

    if name not in param_grid:
        print(f"  Aucun param_grid défini pour {name}, skip.")
        continue

    rs = RandomizedSearchCV(
        estimator           = pipeline,
        param_distributions = param_grid[name],
        n_iter              = n_iter_map.get(name, 20),  # ← adapté par modèle
        scoring             = "neg_mean_absolute_error",
        cv                  = 5,
        n_jobs              = -1,
        random_state        = 42
    )

    rs.fit(X, y)

    print("  Meilleurs paramètres :", rs.best_params_)
    print("  Meilleur MAE CV      :", round(-rs.best_score_, 4))

    results.append({
        "Model"         : name,
        "Best_Params"   : rs.best_params_,
        "Best_CV_MAE"   : round(-rs.best_score_, 4),
        "Best_Estimator": rs.best_estimator_
    })

# Résumé
results_df = (
    pd.DataFrame(results)
    .sort_values("Best_CV_MAE")
    .reset_index(drop=True)
)
print("\n===== Résumé final trié par MAE =====")
print(results_df[["Model", "Best_CV_MAE", "Best_Params"]].to_string(index=False))


===== RandomizedSearchCV pour Ridge =====
  Meilleurs paramètres : {'model__solver': 'svd', 'model__alpha': 0.01}
  Meilleur MAE CV      : 4.2037

===== RandomizedSearchCV pour Lasso =====
  Meilleurs paramètres : {'model__selection': 'random', 'model__max_iter': 10000, 'model__alpha': 0.001}
  Meilleur MAE CV      : 4.2738

===== RandomizedSearchCV pour RandomForest =====
  Meilleurs paramètres : {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 10}
  Meilleur MAE CV      : 3.7242

===== RandomizedSearchCV pour SVR =====
  Meilleurs paramètres : {'model__kernel': 'rbf', 'model__gamma': 'scale', 'model__epsilon': 0.01, 'model__C': 10.0}
  Meilleur MAE CV      : 3.3203

===== RandomizedSearchCV pour KNN =====
  Meilleurs paramètres : {'model__weights': 'distance', 'model__p': 1, 'model__n_neighbors': 7}
  Meilleur MAE CV      : 3.9173

===== RandomizedSearchCV pour Ridge_PCA =====
  Aucun param_g